In [ ]:
import ROOT
%load_ext JupyROOT

In [ ]:
%%cpp
Double_t D0_Integrand(Double_t *x, Double_t *par)
{
    Double_t lambda = x[0];
    
    //Parameters passed via the TF1 object
    Int_t k_obs      = (Int_t)par[0]; //Observed events (17)
    Double_t mu      = par[1]; //Expected background mean (3.8)
    Double_t sigma   = par[2]; // Background uncertainty (0.6)
    
    // 1. Calculate the Poisson tail: P(k >= k_obs | lambda)
    // In ROOT, TMath::PoissonI(k, lambda) calculates (lambda^k * e^-lambda) / k!
    // The sum of probabilities from 0 to k_obs-1 is the complement of our tail.

    Double_t poisson_tail = 1.0;

    for (Int_t i = 0; i < k_obs; ++i)
    {
        poisson_tail -= TMath::PoissonI(i, lambda);
    }
    
    // 2. Calculate the Gaussian probability density for this lambda
    Double_t gaussian = TMath::Gaus(lambda, mu, sigma, kTRUE); // kTRUE normalises the Gaussian
    
    // The integrand is the product of the two
    return poisson_tail * gaussian;
}

In [ ]:
%%cpp
    TF1 *f_integrand = new TF1("f_integrand", D0_Integrand, 0.0, 15.0, 3);
    
    // Set the physical parameters from the 1995 paper (https://arxiv.org/pdf/hep-ex/9503003)
    f_integrand->SetParameter(0, 17); // par[0] = Observed events
    f_integrand->SetParameter(1, 3.8); // par[1] = Expected background mean
    f_integrand->SetParameter(2, 0.6); // par[2] = Systematic uncertainty
    
    // Set parameter names for clarity in ROOT
    f_integrand->SetParNames("k_obs", "mu", "sigma");
    
    // We strictly integrate from 0 to 15 to handle the physical boundary
    Double_t p_value = f_integrand->Integral(0.0, 15.0);

    std::cout << p_value << std::endl;